In [1]:
import os
import gzip
import pickle
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.cosmology import FlatLambdaCDM
from astropy.table import Table, vstack
from astropy.units import Quantity
from IPython.display import display

import slsim
import slsim.Pipelines as pipelines
from slsim.Pipelines import roman_speclite
from slsim.Deflectors.DeflectorPopulation.galaxy_deflectors import GalaxyDeflectors
from slsim.Lenses.LensPopulation.lens_pop import LensPop
from slsim.Sources.SourceCatalogues.QuasarCatalog.quasar_pop import QuasarRate
from slsim.Sources.SourcePopulation.point_plus_extended_sources import PointPlusExtendedSources

warnings.filterwarnings("ignore")

%load_ext autoreload
%autoreload 2

        Use pytest instead. [astropy.tests.runner]
        Use pytest instead. [astropy.utils.decorators]


In [2]:
cosmo = FlatLambdaCDM(H0=70, Om0=0.3)
skypy_config = os.path.join(
    os.path.dirname(os.path.dirname(slsim.__file__)), "data/SkyPy/euclid-roman-lsst-like.yml"
)

roman_speclite.configure_roman_filters()

all_bands = ["F062", "F087", "F106", "F129", "F158", "F184", "F146", "F213", "g", "r", "i", "z", "y", 
             "VIS", "Y", "J", "H"]

base_sky_area = Quantity(value=10, unit="deg2")

## 1. Deflector and source populations

In [3]:
galaxy_pipeline = pipelines.SkyPyPipeline(
    skypy_config=skypy_config, sky_area=base_sky_area, filters=None,
    cosmo=cosmo, z_min=0.0, z_max=3.0,
)
host_galaxy_candidates = vstack(
    [galaxy_pipeline.red_galaxies, galaxy_pipeline.blue_galaxies], join_type="exact"
)
host_galaxy_candidates["galaxy_type"] = np.repeat(
    ["red", "blue"], [len(galaxy_pipeline.red_galaxies), len(galaxy_pipeline.blue_galaxies)]
)

lens_galaxies = GalaxyDeflectors(
    red_galaxy_list=galaxy_pipeline.red_galaxies,
    # blue_galaxy_list=galaxy_pipeline.blue_galaxies,
    kwargs_cut={"band": "i", "band_max": 28, "z_min": 0.01, "z_max": 2.5},
    kwargs_mass2light={}, cosmo=cosmo, sky_area=base_sky_area,
    gamma_pl={"mean": 2.078, "std_dev": 0.16},  # SLACS slope distribution, Auger et al. (2010), Sec. 3.3
)

In [4]:
if os.path.exists("quasar_source_catalog.fits"):
    quasar_source = Table.read("quasar_source_catalog.fits")
else:
    quasar_source = QuasarRate(
        skypy_config=skypy_config, cosmo=cosmo, sky_area=base_sky_area, noise=True,
        redshifts=np.linspace(0.001, 3.01, 100),
        host_galaxy_candidate=host_galaxy_candidates,
        use_qsogen_sed=True, qsogen_bands=all_bands, use_sed_interpolator=True,
    ).quasar_sample(m_min=15, m_max=30, host_galaxy=True)

source_quasar = PointPlusExtendedSources(
    point_plus_extended_sources_list=quasar_source, cosmo=cosmo, sky_area=base_sky_area,
    kwargs_cut={"band": "i", "band_max": 28, "z_min": 0.001, "z_max": 5.0},
    catalog_type="skypy", point_source_type="quasar", extended_source_type="single_sersic",
)

In [5]:
lens_pop_LSST = LensPop(
    deflector_population=lens_galaxies, source_population=source_quasar,
    cosmo=cosmo, sky_area=Quantity(value=200, unit="deg2"),
)


stats_different_speed_factors = {
    "speed_factors": [1, 10, 100, 1000, 10000],
    "total_lenses": [],
    "total_quads": [],
    "total_doubles": [],
    "quad_fraction": [],
}

for speed_factor in stats_different_speed_factors["speed_factors"]:
    print(f"Running LSST lens population with speed factor: {speed_factor}")
    selected_lenses_LSST = lens_pop_LSST.draw_population(
            speed_factor=speed_factor, kwargs_lens_cuts={
                "min_image_separation": 0.5,
                "max_image_separation": 4.0,
                "second_brightest_image_cut": {"i": 23.3},  # LSST i depth, Oguri & Marshall (2010), Table 1
            }, verbose=True
    )

    num_quads = 0
    num_doubles = 0
    for lens in selected_lenses_LSST:
        if lens.image_number[0] == 4:
            num_quads += 1
        elif lens.image_number[0] == 2:
            num_doubles += 1

    stats_different_speed_factors["total_lenses"].append(len(selected_lenses_LSST))
    stats_different_speed_factors["total_quads"].append(num_quads)
    stats_different_speed_factors["total_doubles"].append(num_doubles)
    stats_different_speed_factors["quad_fraction"].append(num_quads / len(selected_lenses_LSST) if len(selected_lenses_LSST) > 0 else 0)


Running LSST lens population with speed factor: 1


Drawing lens population: 100%|██████████| 13390100/13390100 [48:56<00:00, 4560.20it/s]


Running LSST lens population with speed factor: 10


Drawing lens population: 100%|██████████| 1339010/1339010 [05:04<00:00, 4395.25it/s]


Running LSST lens population with speed factor: 100


Drawing lens population: 100%|██████████| 133901/133901 [00:39<00:00, 3427.41it/s]


Running LSST lens population with speed factor: 1000


Drawing lens population: 100%|██████████| 13390/13390 [00:10<00:00, 1336.77it/s]


Running LSST lens population with speed factor: 10000


Drawing lens population: 100%|██████████| 1339/1339 [00:06<00:00, 200.35it/s]


In [6]:
print("Summary of LSST lens population statistics with different speed factors:")
print("{:<15} {:<15} {:<15} {:<15} {:<15}".format(
    "Speed Factor", "Total Lenses", "Total Quads", "Total Doubles", "Quad Fraction"
))
for i, speed_factor in enumerate(stats_different_speed_factors["speed_factors"]):
    print("{:<15} {:<15} {:<15} {:<15} {:<15.4f}".format(
        speed_factor,
        stats_different_speed_factors["total_lenses"][i],
        stats_different_speed_factors["total_quads"][i],
        stats_different_speed_factors["total_doubles"][i],
        stats_different_speed_factors["quad_fraction"][i]
    ))

Summary of LSST lens population statistics with different speed factors:
Speed Factor    Total Lenses    Total Quads     Total Doubles   Quad Fraction  
1               19              0               19              0.0000         
10              17              1               15              0.0588         
100             22              0               22              0.0000         
1000            11              1               9               0.0909         
10000           17              1               15              0.0588         


In [7]:
lens_pop_Euclid = LensPop(
    deflector_population=lens_galaxies, source_population=source_quasar,
    cosmo=cosmo, sky_area=Quantity(value=14500, unit="deg2"),  # Euclid Wide, Scaramella et al. (2022)
)

selected_lenses_Euclid = lens_pop_Euclid.draw_population(
        speed_factor=10000, kwargs_lens_cuts={
            "min_image_separation": 0.2,
            "max_image_separation": 4.0,
            "second_brightest_image_cut": {"VIS": 26.2},  # VIS 5-sigma depth, Scaramella et al. (2022)
        }, verbose=True
)

Drawing lens population: 100%|██████████| 97078/97078 [06:56<00:00, 233.16it/s]
